In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

df_raw = pd.read_csv("Crime_Data_from_2020_to_Present_20260228.csv")
df_raw.head()

Saving Crime_Data_from_2020_to_Present_20260228.csv to Crime_Data_from_2020_to_Present_20260228.csv


,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,...,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,211507896,2021 Apr 11 12:00:00 AM,2020 Nov 07 12:00:00 AM,845,15,N Hollywood,1502,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,7800 BEEMAN AV,NaN,34.2124,-118.4092
1,201516622,2020 Oct 21 12:00:00 AM,2020 Oct 18 12:00:00 AM,1845,15,N Hollywood,1521,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",...,IC,Invest Cont,230.0,NaN,NaN,NaN,ATOLL AV,N GAULT,34.1993,-118.4203
2,240913563,2024 Dec 10 12:00:00 AM,2020 Oct 30 12:00:00 AM,1240,9,Van Nuys,933,2,354,THEFT OF IDENTITY,...,IC,Invest Cont,354.0,NaN,NaN,NaN,14600 SYLVAN ST,NaN,34.1847,-118.4509
3,210704711,2020 Dec 24 12:00:00 AM,2020 Dec 24 12:00:00 AM,1310,7,Wilshire,782,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,...,IC,Invest Cont,331.0,NaN,NaN,NaN,6000 COMEY AV,NaN,34.0339,-118.3747
4,201418201,2020 Oct 03 12:00:00 AM,2020 Sep 29 12:00:00 AM,1830,14,Pacific,1454,1,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),...,IC,Invest Cont,420.0,NaN,NaN,NaN,4700 LA VILLA MARINA,NaN,33.9813,-118.4350


In [ ]:
# Step 1: 查看所有列名
df_raw.columns

Index(['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME',
       'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes',
       'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc',
       'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1',
       'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street', 'LAT',
       'LON'],
      dtype='object')

In [ ]:
# 复制一份数据
df = df_raw.copy()

# 统一列名格式（全部小写 + 下划线 + 去空格）
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.columns

Index(['dr_no', 'date_rptd', 'date_occ', 'time_occ', 'area', 'area_name',
       'rpt_dist_no', 'part_1-2', 'crm_cd', 'crm_cd_desc', 'mocodes',
       'vict_age', 'vict_sex', 'vict_descent', 'premis_cd', 'premis_desc',
       'weapon_used_cd', 'weapon_desc', 'status', 'status_desc', 'crm_cd_1',
       'crm_cd_2', 'crm_cd_3', 'crm_cd_4', 'location', 'cross_street', 'lat',
       'lon'],
      dtype='object')

In [ ]:
df[["date_occ", "crm_cd_desc", "lat", "lon"]].head()

,date_occ,crm_cd_desc,lat,lon
0,2020 Nov 07 12:00:00 AM,THEFT OF IDENTITY,34.2124,-118.4092
1,2020 Oct 18 12:00:00 AM,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",34.1993,-118.4203
2,2020 Oct 30 12:00:00 AM,THEFT OF IDENTITY,34.1847,-118.4509
3,2020 Dec 24 12:00:00 AM,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,34.0339,-118.3747
4,2020 Sep 29 12:00:00 AM,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),33.9813,-118.4350


In [ ]:
# Step 2: 类型转换

# 1️⃣ 转换 date_occ 为 datetime
df["date_occ"] = pd.to_datetime(df["date_occ"], errors="coerce")

# 2️⃣ 转换 lat / lon 为 float
df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
df["lon"] = pd.to_numeric(df["lon"], errors="coerce")

# 3️⃣ 清理 crime 描述（统一大写，去空格）
df["crm_cd_desc"] = df["crm_cd_desc"].astype(str).str.strip().str.upper()

# 查看转换后的类型
df[["date_occ", "lat", "lon"]].dtypes

/tmp/ipykernel_207/110848091.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["date_occ"] = pd.to_datetime(df["date_occ"], errors="coerce")


,0
date_occ,datetime64[ns]
lat,float64
lon,float64


In [ ]:
df[["date_occ", "lat", "lon"]].isna().sum()

,0
date_occ,0
lat,0
lon,0


In [ ]:
# Step 3: 删除无效坐标 + LA bounding box 粗过滤

rows_before = len(df)

# 1) 删除 0 坐标（有些数据会有 0，但你的 NA=0，不代表没有 0）
df = df[(df["lat"] != 0) & (df["lon"] != 0)]

rows_after_zero = len(df)

# 2) LA 粗范围过滤（宁可宽一点，避免误删）
min_lat = 33.2
max_lat = 34.95
min_lon = -119.10
max_lon = -117.40

df = df[(df["lat"] >= min_lat) & (df["lat"] <= max_lat)]
df = df[(df["lon"] >= min_lon) & (df["lon"] <= max_lon)]

rows_after_bbox = len(df)

rows_before, rows_after_zero, rows_after_bbox

(1004991, 1002751, 1002751)

In [ ]:
df[["lat","lon"]].describe()

,lat,lon
count,1.002751e+06,1.002751e+06
mean,3.407416e+01,-1.183547e+02
std,1.111570e-01,1.044454e-01
min,3.370590e+01,-1.186676e+02
25%,3.401540e+01,-1.184309e+02
50%,3.405920e+01,-1.183230e+02
75%,3.416490e+01,-1.182740e+02
max,3.433430e+01,-1.181554e+02


In [ ]:
# Step 4: 筛选 2022–2024 + 添加 year

df["year"] = df["date_occ"].dt.year

df = df[(df["year"] >= 2022) & (df["year"] <= 2024)]

df["year"].value_counts().sort_index()

,count
year,
2022,235258
2023,232341
2024,127567


In [ ]:
# Step 5: 按 dr_no 去重

rows_before = len(df)

df["dr_no"] = df["dr_no"].astype(str).str.strip()
df = df.drop_duplicates(subset=["dr_no"])

rows_after = len(df)

rows_before, rows_after, rows_before - rows_after

(595166, 595166, 0)

In [ ]:
# Step 6: 分类 + 只保留 relevant

violent_keywords = ["HOMICIDE", "ASSAULT", "ROBBERY"]
property_keywords = ["BURGLARY", "THEFT", "CRIMINAL DAMAGE"]

def classify_crime(desc):
    desc = str(desc).upper()
    for k in violent_keywords:
        if k in desc:
            return "violent"
    for k in property_keywords:
        if k in desc:
            return "property"
    return "other"

df["crime_type"] = df["crm_cd_desc"].apply(classify_crime)

df["crime_type"].value_counts()

,count
crime_type,
property,256906
other,208515
violent,129745


In [ ]:
df_rel = df[df["crime_type"] != "other"].copy()

df_rel["crime_type"].value_counts()

,count
crime_type,
property,256906
violent,129745


In [ ]:
df_rel_out = df_rel[[
    "date_occ",
    "year",
    "crime_type",
    "lat",
    "lon"
]].copy()

df_rel_out.to_parquet("crime_points_2022_2024_relevant.parquet", index=False)
df_rel_out.to_csv("crime_points_2022_2024_relevant.csv", index=False)

len(df_rel_out)

386651

In [ ]:
df_rel_out.head()

,date_occ,year,crime_type,lat,lon
351636,2022-09-22,2022,property,34.1052,-118.3252
355413,2022-08-11,2022,property,33.9936,-118.4703
409621,2022-09-15,2022,property,34.2048,-118.4837
409743,2022-09-08,2022,violent,34.0502,-118.2765
409745,2022-07-10,2022,violent,34.2299,-118.5754


In [ ]:
df_rel_out.info()

<class 'pandas.core.frame.DataFrame'>
Index: 386651 entries, 351636 to 1004892
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   date_occ    386651 non-null  datetime64[ns]
 1   year        386651 non-null  int32         
 2   crime_type  386651 non-null  object        
 3   lat         386651 non-null  float64       
 4   lon         386651 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int32(1), object(1)
memory usage: 16.2+ MB


In [ ]:
df_rel_out.describe()

,date_occ,year,lat,lon
count,386651,386651.000000,386651.000000,386651.000000
mean,2023-04-08 20:44:41.227257344,2022.794311,34.077483,-118.358284
min,2022-01-01 00:00:00,2022.000000,33.706400,-118.667600
25%,2022-08-16 00:00:00,2022.000000,34.020300,-118.435200
50%,2023-03-26 00:00:00,2023.000000,34.061100,-118.329400
75%,2023-11-14 00:00:00,2023.000000,34.166700,-118.274700
max,2024-12-30 00:00:00,2024.000000,34.334300,-118.155400
std,NaN,0.749784,0.108694,0.105910


In [ ]:
# Spatial Join

In [ ]:
!pip install geopandas

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving tl_2024_06_tract.shp to tl_2024_06_tract.shp
Saving tl_2024_06_tract.prj to tl_2024_06_tract.prj
Saving tl_2024_06_tract.shx to tl_2024_06_tract.shx
Saving tl_2024_06_tract.dbf to tl_2024_06_tract.dbf


In [ ]:
import geopandas as gpd

tracts = gpd.read_file("tl_2024_06_tract.shp")
tracts.head()

,STATEFP,COUNTYFP,TRACTCE,GEOID,GEOIDFQ,NAME,NAMELSAD,MTFCC,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,06,001,442700,06001442700,1400000US06001442700,4427,Census Tract 4427,G5020,S,1234016,0,+37.5371513,-122.0081095,"POLYGON ((-122.01721 37.53932, -122.01719 37.5..."
1,06,001,442800,06001442800,1400000US06001442800,4428,Census Tract 4428,G5020,S,1278646,0,+37.5293619,-121.9931002,"POLYGON ((-122.0023 37.52984, -122.00224 37.52..."
2,06,037,204920,06037204920,1400000US06037204920,2049.20,Census Tract 2049.20,G5020,S,909972,0,+34.0175004,-118.1974975,"POLYGON ((-118.20284 34.01966, -118.20283 34.0..."
3,06,037,205110,06037205110,1400000US06037205110,2051.10,Census Tract 2051.10,G5020,S,286962,0,+34.0245059,-118.2142985,"POLYGON ((-118.21964 34.02628, -118.21945 34.0..."
4,06,037,205120,06037205120,1400000US06037205120,2051.20,Census Tract 2051.20,G5020,S,1466242,0,+34.0187542,-118.2117951,"POLYGON ((-118.22023 34.02056, -118.22018 34.0..."


In [ ]:
tracts.columns

Index(['STATEFP', 'COUNTYFP', 'TRACTCE', 'GEOID', 'GEOIDFQ', 'NAME',
       'NAMELSAD', 'MTFCC', 'FUNCSTAT', 'ALAND', 'AWATER', 'INTPTLAT',
       'INTPTLON', 'geometry'],
      dtype='object')

In [ ]:
la_tracts = tracts[tracts["COUNTYFP"] == "037"].copy()
la_tracts = la_tracts[["GEOID", "geometry"]]

len(la_tracts)

2498

In [ ]:
la_tracts.crs

<Geographic 2D CRS: EPSG:4269>
Name: NAD83
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: North America - onshore and offshore: Canada - Alberta; British Columbia; Manitoba; New Brunswick; Newfoundland and Labrador; Northwest Territories; Nova Scotia; Nunavut; Ontario; Prince Edward Island; Quebec; Saskatchewan; Yukon. Puerto Rico. United States (USA) - Alabama; Alaska; Arizona; Arkansas; California; Colorado; Connecticut; Delaware; Florida; Georgia; Hawaii; Idaho; Illinois; Indiana; Iowa; Kansas; Kentucky; Louisiana; Maine; Maryland; Massachusetts; Michigan; Minnesota; Mississippi; Missouri; Montana; Nebraska; Nevada; New Hampshire; New Jersey; New Mexico; New York; North Carolina; North Dakota; Ohio; Oklahoma; Oregon; Pennsylvania; Rhode Island; South Carolina; South Dakota; Tennessee; Texas; Utah; Vermont; Virginia; Washington; West Virginia; Wisconsin; Wyoming. US Virgin Islands. British Virgin Islands

In [ ]:
la_tracts = la_tracts.to_crs("EPSG:4326")
la_tracts.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [ ]:
crime_gdf = gpd.GeoDataFrame(
    df_rel_out,
    geometry=gpd.points_from_xy(df_rel_out["lon"], df_rel_out["lat"]),
    crs="EPSG:4326"
)

crime_gdf.head()

,date_occ,year,crime_type,lat,lon,geometry
351636,2022-09-22,2022,property,34.1052,-118.3252,POINT (-118.3252 34.1052)
355413,2022-08-11,2022,property,33.9936,-118.4703,POINT (-118.4703 33.9936)
409621,2022-09-15,2022,property,34.2048,-118.4837,POINT (-118.4837 34.2048)
409743,2022-09-08,2022,violent,34.0502,-118.2765,POINT (-118.2765 34.0502)
409745,2022-07-10,2022,violent,34.2299,-118.5754,POINT (-118.5754 34.2299)


In [ ]:
crime_with_tract = gpd.sjoin(
    crime_gdf,
    la_tracts,
    how="inner",
    predicate="within"
)

crime_with_tract[["date_occ", "year", "crime_type", "lat", "lon", "GEOID"]].head()

,date_occ,year,crime_type,lat,lon,GEOID
351636,2022-09-22,2022,property,34.1052,-118.3252,06037189502
355413,2022-08-11,2022,property,33.9936,-118.4703,06037273300
409621,2022-09-15,2022,property,34.2048,-118.4837,06037127400
409743,2022-09-08,2022,violent,34.0502,-118.2765,06037209520
409745,2022-07-10,2022,violent,34.2299,-118.5754,06037113426


In [ ]:
len(df_rel_out), len(crime_with_tract)

(386651, 386597)

In [ ]:
tract_crime = (
    crime_with_tract
    .groupby(["GEOID", "crime_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

tract_crime["total_relevant"] = tract_crime["property"] + tract_crime["violent"]

tract_crime.head()

crime_type,GEOID,property,violent,total_relevant
0,06037101110,149,65,214
1,06037101122,82,14,96
2,06037101220,128,44,172
3,06037101221,106,37,143
4,06037101222,106,69,175


In [ ]:
tract_crime.to_csv("la_tract_crime_2022_2024_relevant_counts.csv", index=False)
tract_crime.to_parquet("la_tract_crime_2022_2024_relevant_counts.parquet", index=False)

tract_crime.shape

(1206, 4)

In [ ]:
# 把 0 crime tract 补回来

In [ ]:
tract_crime_full = (
    la_tracts[["GEOID"]]
    .merge(tract_crime, on="GEOID", how="left")
    .fillna(0)
)

tract_crime_full.head()

,GEOID,property,violent,total_relevant
0,06037204920,213.0,75.0,288.0
1,06037205110,126.0,76.0,202.0
2,06037205120,574.0,239.0,813.0
3,06037206010,358.0,144.0,502.0
4,06037206020,169.0,213.0,382.0


In [ ]:
tract_crime_full.shape

(2498, 4)

In [ ]:
tract_crime_full[["property", "violent", "total_relevant"]] = \
    tract_crime_full[["property", "violent", "total_relevant"]].astype(int)

tract_crime_full.dtypes

,0
GEOID,object
property,int64
violent,int64
total_relevant,int64


In [ ]:
tract_crime_full.head()

,GEOID,property,violent,total_relevant
0,06037204920,213,75,288
1,06037205110,126,76,202
2,06037205120,574,239,813
3,06037206010,358,144,502
4,06037206020,169,213,382


In [ ]:
# ACS total population

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ACSDT5Y2022.B01003-Data.csv to ACSDT5Y2022.B01003-Data.csv
Saving ACSDT5Y2023.B01003-Data.csv to ACSDT5Y2023.B01003-Data.csv
Saving ACSDT5Y2024.B01003-Data.csv to ACSDT5Y2024.B01003-Data.csv


In [ ]:
import pandas as pd

acs22 = pd.read_csv("ACSDT5Y2022.B01003-Data.csv")
acs23 = pd.read_csv("ACSDT5Y2023.B01003-Data.csv")
acs24 = pd.read_csv("ACSDT5Y2024.B01003-Data.csv")

acs22.head()

,GEO_ID,NAME,B01003_001E,B01003_001M,Unnamed: 4
0,Geography,Geographic Area Name,Estimate!!Total,Margin of Error!!Total,NaN
1,1400000US06037101110,Census Tract 1011.10; Los Angeles County; Cali...,4014,473,NaN
2,1400000US06037101122,Census Tract 1011.22; Los Angeles County; Cali...,4164,822,NaN
3,1400000US06037101220,Census Tract 1012.20; Los Angeles County; Cali...,3481,467,NaN
4,1400000US06037101221,Census Tract 1012.21; Los Angeles County; Cali...,3756,687,NaN


In [ ]:
acs22.columns

Index(['GEO_ID', 'NAME', 'B01003_001E', 'B01003_001M', 'Unnamed: 4'], dtype='object')

In [ ]:
acs22 = acs22.iloc[1:].copy()
acs23 = acs23.iloc[1:].copy()
acs24 = acs24.iloc[1:].copy()

acs22.head()

,GEO_ID,NAME,B01003_001E,B01003_001M,Unnamed: 4
1,1400000US06037101110,Census Tract 1011.10; Los Angeles County; Cali...,4014,473,NaN
2,1400000US06037101122,Census Tract 1011.22; Los Angeles County; Cali...,4164,822,NaN
3,1400000US06037101220,Census Tract 1012.20; Los Angeles County; Cali...,3481,467,NaN
4,1400000US06037101221,Census Tract 1012.21; Los Angeles County; Cali...,3756,687,NaN
5,1400000US06037101222,Census Tract 1012.22; Los Angeles County; Cali...,2808,424,NaN


In [ ]:
acs22["GEOID"] = acs22["GEO_ID"].astype(str).str[-11:]
acs23["GEOID"] = acs23["GEO_ID"].astype(str).str[-11:]
acs24["GEOID"] = acs24["GEO_ID"].astype(str).str[-11:]

In [ ]:
acs22["pop_2022"] = pd.to_numeric(acs22["B01003_001E"], errors="coerce")
acs23["pop_2023"] = pd.to_numeric(acs23["B01003_001E"], errors="coerce")
acs24["pop_2024"] = pd.to_numeric(acs24["B01003_001E"], errors="coerce")

In [ ]:
pop22 = acs22[["GEOID", "pop_2022"]].copy()
pop23 = acs23[["GEOID", "pop_2023"]].copy()
pop24 = acs24[["GEOID", "pop_2024"]].copy()

In [ ]:
acs_pop = pop22.merge(pop23, on="GEOID", how="outer")
acs_pop = acs_pop.merge(pop24, on="GEOID", how="outer")

acs_pop.head()

,GEOID,pop_2022,pop_2023,pop_2024
0,06037101110,4014,4152,4294
1,06037101122,4164,4198,4124
2,06037101220,3481,3434,3467
3,06037101221,3756,3931,3593
4,06037101222,2808,2572,2290


In [ ]:
acs_pop_la = acs_pop[acs_pop["GEOID"].str.startswith("06037")].copy()

acs_pop_la.shape

(2498, 4)

In [ ]:
acs_pop_la["pop_avg_22_24"] = (
    acs_pop_la["pop_2022"] +
    acs_pop_la["pop_2023"] +
    acs_pop_la["pop_2024"]
) / 3

acs_pop_la.head()

,GEOID,pop_2022,pop_2023,pop_2024,pop_avg_22_24
0,06037101110,4014,4152,4294,4153.333333
1,06037101122,4164,4198,4124,4162.000000
2,06037101220,3481,3434,3467,3460.666667
3,06037101221,3756,3931,3593,3760.000000
4,06037101222,2808,2572,2290,2556.666667


In [ ]:
crime_acs = tract_crime_full.merge(
    acs_pop_la[["GEOID", "pop_avg_22_24"]],
    on="GEOID",
    how="left"
)

crime_acs.head()

,GEOID,property,violent,total_relevant,pop_avg_22_24
0,06037204920,213,75,288,2452.000000
1,06037205110,126,76,202,3546.000000
2,06037205120,574,239,813,3430.333333
3,06037206010,358,144,502,3537.000000
4,06037206020,169,213,382,8267.333333


In [ ]:
crime_acs["violent_per_1000"] = crime_acs["violent"] / crime_acs["pop_avg_22_24"] * 1000
crime_acs["property_per_1000"] = crime_acs["property"] / crime_acs["pop_avg_22_24"] * 1000

crime_acs[["GEOID", "violent_per_1000", "property_per_1000"]].head()

,GEOID,violent_per_1000,property_per_1000
0,06037204920,30.587276,86.867863
1,06037205110,21.432600,35.532995
2,06037205120,69.672529,167.330677
3,06037206010,40.712468,101.215720
4,06037206020,25.764051,20.441900


In [ ]:
crime_acs["pop_avg_22_24"].isna().sum()

np.int64(0)

In [ ]:
crime_acs["pop_avg_22_24"].describe()

,pop_avg_22_24
count,2498.000000
mean,3948.994262
std,1430.200930
min,0.000000
25%,2974.166667
50%,3885.000000
75%,4828.583333
max,13154.000000


In [ ]:
crime_acs_valid = crime_acs[crime_acs["pop_avg_22_24"] >= 500].copy()

crime_acs_valid.shape

(2463, 7)

In [ ]:
violent_cut = crime_acs_valid["violent_per_1000"].quantile(0.60)
property_cut = crime_acs_valid["property_per_1000"].quantile(0.60)

violent_cut, property_cut

(np.float64(8.858695809394872), np.float64(26.587723222139015))

In [ ]:
crime_acs_valid["crime_ok_60"] = (
    (crime_acs_valid["violent_per_1000"] <= violent_cut) &
    (crime_acs_valid["property_per_1000"] <= property_cut)
)

crime_acs_valid["crime_ok_60"].value_counts()

,count
crime_ok_60,
True,1394
False,1069


In [ ]:
import pandas as pd

def acs_clean_one_year(df_raw, year, value_col, new_col_name):
    """
    Extract GEOID and one ACS estimate column from a raw ACS file.
    """
    df = df_raw.iloc[1:].copy()

    # 1) GEOID: Extract the last 11 digits from GEO_ID (tract ID)
    df["GEOID"] = df["GEO_ID"].astype(str).str[-11:]

    # 2) Convert value column to numeric
    df[new_col_name] = pd.to_numeric(df[value_col], errors="coerce")

    # 3) Keep only required columns
    df = df[["GEOID", new_col_name]].copy()

    return df

In [ ]:
def acs_merge_3years(df22_raw, df23_raw, df24_raw, value_col, base_name):
    """
    Merge 2022/2023/2024 of the same ACS table into a wide format:
    GEOID | {base_name}_2022 | {base_name}_2023 | {base_name}_2024 | {base_name}_avg_22_24
    """
    d22 = acs_clean_one_year(df22_raw, 2022, value_col, f"{base_name}_2022")
    d23 = acs_clean_one_year(df23_raw, 2023, value_col, f"{base_name}_2023")
    d24 = acs_clean_one_year(df24_raw, 2024, value_col, f"{base_name}_2024")

    out = d22.merge(d23, on="GEOID", how="outer")
    out = out.merge(d24, on="GEOID", how="outer")

    out[f"{base_name}_avg_22_24"] = (
        out[f"{base_name}_2022"] +
        out[f"{base_name}_2023"] +
        out[f"{base_name}_2024"]
    ) / 3

    # Keep only LA County tracts (tracts starting with 06037)
    out = out[out["GEOID"].astype(str).str.startswith("06037")].copy()

    return out

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving ACSDT5Y2024.B15003-Data.csv to ACSDT5Y2024.B15003-Data.csv
Saving ACSDT5Y2024.B19013-Data.csv to ACSDT5Y2024.B19013-Data.csv
Saving ACSDT5Y2023.B19013-Data.csv to ACSDT5Y2023.B19013-Data.csv
Saving ACSDT5Y2022.B19013-Data.csv to ACSDT5Y2022.B19013-Data.csv
Saving ACSDT5Y2022.B15003-Data.csv to ACSDT5Y2022.B15003-Data.csv
Saving ACSDT5Y2023.B15003-Data.csv to ACSDT5Y2023.B15003-Data.csv


In [ ]:
inc22 = pd.read_csv("ACSDT5Y2022.B19013-Data.csv")
inc23 = pd.read_csv("ACSDT5Y2023.B19013-Data.csv")
inc24 = pd.read_csv("ACSDT5Y2024.B19013-Data.csv")

In [ ]:
acs_income_la = acs_merge_3years(
    inc22,
    inc23,
    inc24,
    value_col="B19013_001E",
    base_name="income"
)

acs_income_la.head()

,GEOID,income_2022,income_2023,income_2024,income_avg_22_24
0,06037101110,68972.0,84091.0,88438.0,80500.333333
1,06037101122,118859.0,99583.0,104643.0,107695.000000
2,06037101220,65139.0,69676.0,92317.0,75710.666667
3,06037101221,53348.0,53798.0,52204.0,53116.666667
4,06037101222,36779.0,45662.0,38125.0,40188.666667


In [ ]:
crime_features = crime_acs_valid.merge(
    acs_income_la[["GEOID", "income_avg_22_24"]],
    on="GEOID",
    how="left"
)

crime_features.head()

,GEOID,property,violent,total_relevant,pop_avg_22_24,violent_per_1000,property_per_1000,crime_ok,crime_ok_60,income_avg_22_24
0,06037204920,213,75,288,2452.000000,30.587276,86.867863,False,False,71185.000000
1,06037205110,126,76,202,3546.000000,21.432600,35.532995,True,False,52863.333333
2,06037205120,574,239,813,3430.333333,69.672529,167.330677,False,False,30289.333333
3,06037206010,358,144,502,3537.000000,40.712468,101.215720,False,False,62886.666667
4,06037206020,169,213,382,8267.333333,25.764051,20.441900,False,False,140919.000000


In [ ]:
# education

In [ ]:
edu22 = pd.read_csv("ACSDT5Y2022.B15003-Data.csv")
edu23 = pd.read_csv("ACSDT5Y2023.B15003-Data.csv")
edu24 = pd.read_csv("ACSDT5Y2024.B15003-Data.csv")

In [ ]:
def acs_clean_ba_plus_rate(df_raw, year, new_col_name):
    df = df_raw.iloc[1:].copy()

    df["GEOID"] = df["GEO_ID"].astype(str).str[-11:]

    total_25 = pd.to_numeric(df["B15003_001E"], errors="coerce")

    ba = pd.to_numeric(df["B15003_022E"], errors="coerce")
    ma = pd.to_numeric(df["B15003_023E"], errors="coerce")
    prof = pd.to_numeric(df["B15003_024E"], errors="coerce")
    phd = pd.to_numeric(df["B15003_025E"], errors="coerce")

    ba_plus = ba + ma + prof + phd

    df[new_col_name] = ba_plus / total_25

    df = df[["GEOID", new_col_name]].copy()

    return df

In [ ]:
edu_22 = acs_clean_ba_plus_rate(edu22, 2022, "ba_plus_rate_2022")
edu_23 = acs_clean_ba_plus_rate(edu23, 2023, "ba_plus_rate_2023")
edu_24 = acs_clean_ba_plus_rate(edu24, 2024, "ba_plus_rate_2024")

acs_edu_la = edu_22.merge(edu_23, on="GEOID", how="outer")
acs_edu_la = acs_edu_la.merge(edu_24, on="GEOID", how="outer")

acs_edu_la["ba_plus_rate_avg_22_24"] = (
    acs_edu_la[
        ["ba_plus_rate_2022",
         "ba_plus_rate_2023",
         "ba_plus_rate_2024"]
    ].mean(axis=1)
)

acs_edu_la = acs_edu_la[acs_edu_la["GEOID"].astype(str).str.startswith("06037")].copy()

acs_edu_la.head()

,GEOID,ba_plus_rate_2022,ba_plus_rate_2023,ba_plus_rate_2024,ba_plus_rate_avg_22_24
0,06037101110,0.289195,0.274950,0.287255,0.283800
1,06037101122,0.384100,0.336900,0.380623,0.367207
2,06037101220,0.289453,0.314072,0.367477,0.323667
3,06037101221,0.272185,0.237264,0.277820,0.262423
4,06037101222,0.120574,0.190999,0.166191,0.159255


In [ ]:
crime_features = crime_features.merge(
    acs_edu_la[["GEOID", "ba_plus_rate_avg_22_24"]],
    on="GEOID",
    how="left"
)

In [ ]:
crime_features = crime_features.loc[:, ~crime_features.T.duplicated()]

In [ ]:
crime_features = crime_features.rename(
    columns={"ba_plus_rate_avg_22_24_x": "ba_plus_rate_avg_22_24"}
)

In [ ]:
crime_features.head()

,GEOID,property,violent,total_relevant,pop_avg_22_24,violent_per_1000,property_per_1000,crime_ok,crime_ok_60,income_avg_22_24,ba_plus_rate_avg_22_24
0,06037204920,213,75,288,2452.000000,30.587276,86.867863,False,False,71185.000000,0.070387
1,06037205110,126,76,202,3546.000000,21.432600,35.532995,True,False,52863.333333,0.063501
2,06037205120,574,239,813,3430.333333,69.672529,167.330677,False,False,30289.333333,0.067817
3,06037206010,358,144,502,3537.000000,40.712468,101.215720,False,False,62886.666667,0.377261
4,06037206020,169,213,382,8267.333333,25.764051,20.441900,False,False,140919.000000,0.069375
